# 01 Data Cleaning

This notebook prepares the Salifort Motors HR dataset for analysis and modeling. The goal is to create a reliable processed dataset with consistent column names, duplicate handling, and a documented data quality review.

**Business context:** HR stakeholders need a trustworthy analytical base before drawing conclusions about employee attrition risk.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)

In [ ]:
from preprocessing import clean_employee_data, summarize_data_quality
from utils import ensure_project_dirs, load_raw_data, save_processed_data

ensure_project_dirs()

## Load Raw Data

Place `HR_capstone_dataset.csv` in `data/raw/` before running this notebook. The file is intentionally not committed because raw data files can be large or externally sourced.

In [ ]:
raw_df = load_raw_data()
raw_df.head()

In [ ]:
print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
raw_df.info()

## Standardize Columns and Remove Duplicate Records

The original capstone data uses mixed naming conventions such as `Work_accident` and `time_spend_company`. These are standardized to snake_case, and `time_spend_company` is renamed to `tenure` for readability.

In [ ]:
clean_df = clean_employee_data(raw_df, drop_duplicates=True)
print(f"Clean shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]:,} columns")
print(f"Duplicate rows removed: {raw_df.duplicated().sum():,}")
clean_df.head()

## Data Quality Summary

This table confirms missingness, data types, and cardinality for each feature. A clean summary helps recruiters and stakeholders see that the modeling workflow starts with basic data validation.

In [ ]:
quality_summary = summarize_data_quality(clean_df)
quality_summary

In [ ]:
clean_df.describe(include="all").T

## Outlier Review

Tenure outliers are reviewed but not automatically removed. Tree-based models are less sensitive to outliers, and long-tenure employees may represent real retention behavior that HR should understand.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
sns.boxplot(data=clean_df, x="tenure", ax=ax)
ax.set_title("Tenure Distribution")
ax.set_xlabel("Years at company")
plt.show()

q1 = clean_df["tenure"].quantile(0.25)
q3 = clean_df["tenure"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outlier_count = (clean_df["tenure"] > upper_bound).sum()
print(f"Upper outlier threshold: {upper_bound:.1f} years")
print(f"Tenure outlier rows: {outlier_count:,}")

## Save Processed Dataset

The cleaned dataset is saved for the downstream EDA and modeling notebooks.

In [ ]:
processed_path = save_processed_data(clean_df)
processed_path